# Model Comparison Dashboard

Final quantitative head-to-head comparison of all methods:

| Metric | Meaning |
|---|---|
| Log-Likelihood | How well the model fits the data |
| AIC / BIC | Penalised fit : lower is better |
| Regime Persistence | Mean days per regime (higher = more stable) |
| Purity vs Vol | Alignment with observable vol regimes |
| Conditional Sharpe | Risk-adjusted return per regime |
| Wasserstein separation | Distribution spread between states |

---

## 0. Setup

In [16]:
import sys, os
project_root = os.path.abspath("..")
sys.path.insert(0, project_root)

scratch_path = os.path.join(project_root, "01_hmm_from_scratch")
sys.path.insert(0, scratch_path)

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from scipy.stats import norm, wasserstein_distance
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from hmmlearn import hmm as hmmlearn_hmm
import warnings
warnings.filterwarnings("ignore")

from utils.data_utils import prepare_ticker_data, get_observation_sequence, time_split
from utils.metrics    import (regime_statistics, regime_persistence, regime_purity,
                               pairwise_wasserstein, aic, bic, hmm_n_params,
                               regime_persistence_summary, model_comparison_table)
from utils.viz_utils  import (plot_price_with_regimes, plot_regime_distributions,
                               plot_transition_matrix, REGIME_COLOURS, _DARK_LAYOUT)
from hmm_core import GaussianHMM

print("All imports OK")

All imports OK


In [17]:
TICKER = "SPY"
df = prepare_ticker_data(TICKER, start="2024-01-01")
df_train, df_test = time_split(df, train_frac=0.80)

FEATURE_COLS = ["Returns", "AbsReturn", "RVol_20d", "RVol_60d", "Momentum_5d"]
X_1d_train = df_train["Returns"].values
X_1d_test  = df_test["Returns"].values
scaler = StandardScaler()
X_sc_train = scaler.fit_transform(df_train[FEATURE_COLS])
X_sc_test  = scaler.transform(df_test[FEATURE_COLS])
vol_labels = df_train["VolRegime"].values

print(f"Train: {len(X_1d_train)}  Test: {len(X_1d_test)}")

Train: 364  Test: 92


## 1. Fit All Models

In [18]:
# HMM (scratch)
hmm_sc = GaussianHMM(n_states=3, n_iter=300, tol=1e-7, random_state=42)
hmm_sc.fit(X_1d_train)
labels_hmm_sc = hmm_sc.predict(X_1d_train)
ll_hmm_sc     = hmm_sc.log_likelihood(X_1d_train)

# HMM (hmmlearn)
h3 = hmmlearn_hmm.GaussianHMM(n_components=3, covariance_type="full",
                                n_iter=300, tol=1e-7, random_state=42)
h3.fit(X_1d_train.reshape(-1,1))
stds_h3 = np.sqrt(h3.covars_.ravel())
order_h3 = np.argsort(stds_h3)
remap_h3 = {order_h3[k]:k for k in range(3)}
labels_hmm_hl = np.array([remap_h3[l] for l in h3.predict(X_1d_train.reshape(-1,1))])
ll_hmm_hl     = h3.score(X_1d_train.reshape(-1,1)) * len(X_1d_train)

# 2-state HMM
h2 = hmmlearn_hmm.GaussianHMM(n_components=2, covariance_type="full",
                                n_iter=300, tol=1e-7, random_state=42)
h2.fit(X_1d_train.reshape(-1,1))
stds_h2 = np.sqrt(h2.covars_.ravel())
labels_hmm_2 = np.array([np.argsort(stds_h2).tolist().index(l)
                           for l in h2.predict(X_1d_train.reshape(-1,1))])
ll_hmm_2 = h2.score(X_1d_train.reshape(-1,1)) * len(X_1d_train)

# GMM
g3 = GaussianMixture(n_components=3, covariance_type="full", random_state=42, n_init=10)
g3.fit(X_1d_train.reshape(-1,1))
stds_g3 = np.sqrt(g3.covariances_.ravel())
order_g3 = np.argsort(stds_g3)
remap_g3 = {order_g3[k]:k for k in range(3)}
labels_gmm = np.array([remap_g3[l] for l in g3.predict(X_1d_train.reshape(-1,1))])
# GMM LL
ll_gmm = g3.score(X_1d_train.reshape(-1,1)) * len(X_1d_train)

# K-Means
km3 = KMeans(n_clusters=3, random_state=42, n_init=20)
labels_km = km3.fit_predict(X_sc_train)
stds_km   = [X_1d_train[labels_km == k].std() for k in range(3)]
order_km  = np.argsort(stds_km)
remap_km  = {order_km[k]:k for k in range(3)}
labels_km = np.array([remap_km[l] for l in labels_km])

vol_map = {"Low": 0, "Med": 1, "High": 2}
labels_obs = np.array([vol_map[v] for v in vol_labels])

print("All models fitted")

Model is not converging.  Current: 1177.401840253265 is not greater than 1177.4372662197331. Delta is -0.03542596646821039


  Converged at iteration 127  (ΔLL = 9.87e-08)
All models fitted


## 2. Quantitative Comparison Table

In [8]:
T = len(X_1d_train)
k3 = hmm_n_params(3)
k2 = hmm_n_params(2)

def mean_persist(labels):
    p = regime_persistence(labels)
    return np.mean(list(p.values()))

def avg_wass(labels, ret):
    W = pairwise_wasserstein(ret, labels)
    vals = W.values.flatten()
    return vals[vals > 0].mean()

rows = [
    {"Model": "HMM 3-state (scratch)", "LL Train": ll_hmm_sc,
     "AIC": aic(ll_hmm_sc, k3), "BIC": bic(ll_hmm_sc, k3, T),
     "Persist": mean_persist(labels_hmm_sc),
     "Purity": regime_purity(labels_hmm_sc, labels_obs),
     "Avg W-dist": avg_wass(labels_hmm_sc, X_1d_train)},
    {"Model": "HMM 3-state (hmmlearn)", "LL Train": ll_hmm_hl,
     "AIC": aic(ll_hmm_hl, k3), "BIC": bic(ll_hmm_hl, k3, T),
     "Persist": mean_persist(labels_hmm_hl),
     "Purity": regime_purity(labels_hmm_hl, labels_obs),
     "Avg W-dist": avg_wass(labels_hmm_hl, X_1d_train)},
    {"Model": "HMM 2-state (hmmlearn)", "LL Train": ll_hmm_2,
     "AIC": aic(ll_hmm_2, k2), "BIC": bic(ll_hmm_2, k2, T),
     "Persist": mean_persist(labels_hmm_2),
     "Purity": regime_purity(labels_hmm_2, labels_obs),
     "Avg W-dist": avg_wass(labels_hmm_2, X_1d_train)},
    {"Model": "GMM 3-component", "LL Train": ll_gmm,
     "AIC": aic(ll_gmm, k3), "BIC": bic(ll_gmm, k3, T),
     "Persist": mean_persist(labels_gmm),
     "Purity": regime_purity(labels_gmm, labels_obs),
     "Avg W-dist": avg_wass(labels_gmm, X_1d_train)},
    {"Model": "K-Means (3 clusters)", "LL Train": float("nan"),
     "AIC": float("nan"), "BIC": float("nan"),
     "Persist": mean_persist(labels_km),
     "Purity": regime_purity(labels_km, labels_obs),
     "Avg W-dist": avg_wass(labels_km, X_1d_train)},
    {"Model": "Observable (vol buckets)", "LL Train": float("nan"),
     "AIC": float("nan"), "BIC": float("nan"),
     "Persist": mean_persist(labels_obs),
     "Purity": 1.0,
     "Avg W-dist": avg_wass(labels_obs, X_1d_train)},
]

comp_df = pd.DataFrame(rows)
comp_df = comp_df.round({"LL Train":1,"AIC":1,"BIC":1,"Persist":1,"Purity":3,"Avg W-dist":6})
print(comp_df.to_string(index=False))

                   Model  LL Train       AIC       BIC  Persist  Purity  Avg W-dist
   HMM 3-state (scratch)    1212.4   -2394.9   -2336.4     10.8   0.560    0.029977
  HMM 3-state (hmmlearn)  423699.3 -847368.5 -847310.1      2.7   0.420    0.028472
  HMM 2-state (hmmlearn)  428569.4 -857122.8 -857091.6     92.5   0.420    0.042029
         GMM 3-component    1196.9   -2363.7   -2305.3      4.1   0.420    0.077264
    K-Means (3 clusters)       NaN       NaN       NaN      8.3   0.462    0.018273
Observable (vol buckets)       NaN       NaN       NaN     12.5   1.000    0.003725


In [9]:
# Interactive comparison table
header = dict(values=list(comp_df.columns),
              fill_color="rgba(40,40,60,1)", font=dict(color="white", size=12),
              align="left")
cells_data = dict(values=[comp_df[c].astype(str) for c in comp_df.columns],
                  fill_color="rgba(20,22,30,1)", font=dict(color="#e0e0e0", size=11),
                  align="left", height=30)
fig = go.Figure(go.Table(header=header, cells=cells_data))
fig.update_layout(title="Model Comparison Summary", height=400,
                  paper_bgcolor="rgba(15,17,22,1)", font=dict(color="#e0e0e0"))
fig.show()

## 3. AIC / BIC Comparison

In [10]:
# Models with valid AIC/BIC
aic_models = comp_df[comp_df["AIC"].apply(lambda x: str(x) != "nan")].copy()
aic_vals   = aic_models["AIC"].astype(float).tolist()
bic_vals   = aic_models["BIC"].astype(float).tolist()
names      = aic_models["Model"].tolist()

from utils.viz_utils import plot_aic_bic
fig = plot_aic_bic(names, aic_vals, bic_vals)
fig.show()

## 4. Regime Persistence Comparison

In [11]:
all_labels = {
    "HMM 3-state (scratch)": labels_hmm_sc,
    "HMM 3-state (hmmlearn)": labels_hmm_hl,
    "GMM 3-component": labels_gmm,
    "K-Means": labels_km,
    "Observable (vol)": labels_obs,
}

fig = go.Figure()
colours_bar = ["rgba(99,200,180,0.8)","rgba(255,195,55,0.8)","rgba(240,80,80,0.8)",
                "rgba(140,100,240,0.8)","rgba(80,160,240,0.8)"]
for idx, (name, lab) in enumerate(all_labels.items()):
    p = regime_persistence(lab)
    means = [p.get(k, 0) for k in sorted(p.keys())]
    fig.add_trace(go.Bar(
        name=name,
        x=[f"State {k}" for k in sorted(p.keys())],
        y=means,
        marker_color=colours_bar[idx % len(colours_bar)],
    ))

fig.update_layout(title="Mean Regime Persistence by State (days)",
                  barmode="group", height=420, **_DARK_LAYOUT)
fig.show()

## 5. Regime Statistics

In [12]:
for name, lab in all_labels.items():
    print(f"\n{'='*55}")
    print(f" {name}")
    print('='*55)
    stats = regime_statistics(X_1d_train, lab)
    print(stats.round(4).to_string(index=False))


 HMM 3-state (scratch)
 State  Mean Return (ann.)  Volatility (ann.)  Sharpe (ann.)  Skewness  Excess Kurtosis  # Days  % of Sample
     0              0.4923             0.0771         6.3870   -0.0520          -0.0533     207      56.8681
     1             -0.1468             0.1940        -0.7567   -0.0762          -0.2483     151      41.4835
     2             -3.0819             0.8580        -3.5918    1.7451           3.2341       6       1.6484

 HMM 3-state (hmmlearn)
 State  Mean Return (ann.)  Volatility (ann.)  Sharpe (ann.)  Skewness  Excess Kurtosis  # Days  % of Sample
     0              0.0959             0.1333         0.7197   -0.8790           1.3946     179      49.1758
     1              0.3496             0.1467         2.3830   -0.1249           1.5995     179      49.1758
     2             -3.0819             0.8580        -3.5918    1.7451           3.2341       6       1.6484

 GMM 3-component
 State  Mean Return (ann.)  Volatility (ann.)  Sharpe (ann.) 

## 6. Purity vs Observable Regimes

In [13]:
purity_vals = [regime_purity(lab, labels_obs) for lab in all_labels.values()]
model_names = list(all_labels.keys())

fig = go.Figure(go.Bar(
    x=model_names, y=purity_vals,
    marker_color="rgba(99,200,180,0.8)",
    text=[f"{v:.3f}" for v in purity_vals],
    textposition="outside",
))
fig.update_layout(title="Clustering Purity vs Observable Vol-Regime Labels",
                  yaxis_title="Purity (higher = better)", yaxis_range=[0, 1.15],
                  height=400, **_DARK_LAYOUT)
fig.show()

## 7. Conditional Sharpe Ratios per State

In [14]:
from utils.metrics import conditional_sharpe

fig = go.Figure()
for idx, (name, lab) in enumerate(all_labels.items()):
    sharpes = conditional_sharpe(X_1d_train, lab)
    states  = sorted(sharpes.keys())
    vals    = [sharpes[s] for s in states]
    fig.add_trace(go.Bar(
        name=name,
        x=[f"State {s}" for s in states],
        y=vals,
        marker_color=colours_bar[idx % len(colours_bar)],
    ))
fig.add_hline(y=0, line_dash="dash", line_color="rgba(200,200,200,0.5)")
fig.update_layout(title="Conditional Annualised Sharpe Ratio per State",
                  barmode="group", height=450, **_DARK_LAYOUT)
fig.show()

## 8. Regime Overlay — All Methods on Same Chart

In [15]:
fig = make_subplots(rows=len(all_labels)+1, cols=1, shared_xaxes=True,
                    subplot_titles=["SPY Price"] + list(all_labels.keys()),
                    row_heights=[0.3] + [0.14]*len(all_labels))

fig.add_trace(go.Scatter(x=df_train.index, y=df_train["Close"],
    line=dict(color="rgba(150,180,255,0.9)", width=1.2), name="SPY"), row=1, col=1)

for row_idx, (name, lab) in enumerate(all_labels.items(), start=2):
    fig.add_trace(go.Scatter(
        x=df_train.index, y=lab.astype(float),
        name=name, mode="lines",
        line=dict(color=colours_bar[(row_idx-2)%5], width=0.9),
        fill="tozeroy",
        fillcolor=colours_bar[(row_idx-2)%5].replace("0.8","0.2"),
    ), row=row_idx, col=1)
    fig.update_yaxes(title_text=name[:15], tickvals=[0,1,2],
                     ticktext=["L","M","H"], row=row_idx, col=1)

fig.update_layout(title="All Regime Detection Methods — Side-by-Side Overlay",
                  height=1000, showlegend=False, **_DARK_LAYOUT)
fig.update_xaxes(showgrid=True, gridcolor="rgba(80,80,80,0.3)")
fig.update_yaxes(showgrid=False)
fig.show()

## 9. Final Recommendations

### Which model should you use?

**For research / quantitative finance:**
> HMM (hmmlearn) - best combination of temporal coherence, probabilistic framework, and computational speed.  
> Use 3 states. Validate on out-of-sample log-likelihood and regime stability.

**For anomaly/crisis detection:**
> Isolation Forest - precisely isolates tail events without imposing state structure.

**For distribution-aware analysis:**
> Wasserstein Clustering - captures distributional shifts that mean/variance methods miss.

**For fast exploratory analysis:**
> GMM - fits in seconds, clean probabilistic output, easy to interpret.

**For supervised regime prediction:**
> Random Forest - use when historical regime labels are available.